# 03 — Mitigation Experiments

This notebook evaluates mitigation strategies:
1. Query rewriting effectiveness
2. Chunk reranking impact
3. Self-reflection loop analysis
4. Before/after score comparisons

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import pandas as pd
from tqdm import tqdm

from src.utils.data_loader import load_halueval
from src.detection.ensemble import EnsembleDetector
from src.rag_pipeline.generator import LLMGenerator
from src.mitigation.self_reflection import SelfReflectionMitigator
from src.mitigation.query_rewriter import QueryRewriter
from src.mitigation.reranker import ChunkReranker
from src.evaluation.metrics import mitigation_improvement_rate

In [ ]:
# Load hallucinated samples only
data = load_halueval(split='test')
hallucinated_samples = [s for s in data if s['label'] == 1][:50]
print(f'Hallucinated samples for mitigation: {len(hallucinated_samples)}')

## 1. Query Rewriting

In [ ]:
rewriter = QueryRewriter()

# Test query rewriting on a sample
sample = hallucinated_samples[0]
rewrites = rewriter.rewrite(
    original_query=sample['query'],
    failed_response=sample['response'],
)
print(f'Original: {sample["query"]}')
print(f'Rewrites:')
for i, r in enumerate(rewrites):
    print(f'  {i+1}. {r}')

## 2. Chunk Reranking

In [ ]:
reranker = ChunkReranker()

# Test reranking on a sample with context
sample = hallucinated_samples[0]
if sample['context']:
    # Simulate multiple chunks
    chunks = sample['context'][0].split('. ')
    ranked = reranker.rerank(sample['query'], chunks, top_k=3)
    print('Top 3 reranked chunks:')
    for i, c in enumerate(ranked):
        print(f'  {i+1}. {c}')

## 3. Self-Reflection Loop

In [ ]:
detector = EnsembleDetector()
generator = LLMGenerator()
mitigator = SelfReflectionMitigator(max_iterations=3)

# Run self-reflection on hallucinated samples
pre_scores = []
post_scores = []

for sample in tqdm(hallucinated_samples[:10], desc='Self-reflection'):
    # Get initial score
    pre_result = detector.score(
        query=sample['query'],
        response=sample['response'],
        context_chunks=sample['context'],
    )
    pre_scores.append(pre_result['final_score'])
    
    # Apply mitigation
    mit_result = mitigator.mitigate(
        query=sample['query'],
        hallucinated_response=sample['response'],
        context_chunks=sample['context'],
        detector=detector,
        generator=generator,
    )
    post_scores.append(mit_result['final_score'])

improvement = mitigation_improvement_rate(pre_scores, post_scores)
print(f'\nMean improvement rate: {improvement:.4f}')
print(f'Mean pre-score: {np.mean(pre_scores):.4f}')
print(f'Mean post-score: {np.mean(post_scores):.4f}')

## 4. Before/After Visualization

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Pre-Mitigation Score': pre_scores,
    'Post-Mitigation Score': post_scores,
    'Improvement': [pre - post for pre, post in zip(pre_scores, post_scores)],
})
print(comparison.describe())